# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd
from pathlib import Path

# Use the repository's starter dataset; do not alter the data.
DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")
assert DATA_PATH.exists(), (
    "Starter CSV not found — run this notebook from the repository root."
)
df = pd.read_csv(DATA_PATH)

fields = [
    "impressions_90d",
    "clicks_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

missing_fields = sorted(set(fields) - set(df.columns))
assert not missing_fields, f"Missing expected columns: {missing_fields}"

# n is the non-missing count; missing is shown separately.
summary = pd.DataFrame({
    "n (non-missing)": df[fields].count(),
    "missing": df[fields].isna().sum(),
    "min": df[fields].min(),
    "median": df[fields].median(),
    "mean": df[fields].mean(),
    "max": df[fields].max(),
})
summary.index.name = "field"

quantiles = df[fields].quantile([0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).T
quantiles.columns = ["p25", "p50", "p75", "p90", "p95", "p99"]
quantiles.index.name = "field"

print("Distribution summary")
print(summary.round(2).to_string())
print("\nQuantiles (use the upper percentiles to spot long or heavy tails)")
print(quantiles.round(2).to_string())

zero_position_rows = (df["avg_position"] == 0).sum()
print(
    f"avg_position == 0: {zero_position_rows:,} rows "
    "(0 means no position data, not rank zero)."
)


## 2. Signal test #1 / #2 / #3 (verdict each)

### Signal test 1 — Staleness / refresh: `days_since_last_update`

This measures how long a page has gone without an update. The table checks whether decline rates differ across refresh-age buckets. Observed pattern: the rate rises from 51.1% (0–30 days) to 61.1% (91–180 days), then is 47.1% for 181+ days; the 31–90-day and 181+-day buckets each have fewer than 200 pages.

Verdict: ______

### Signal test 2 — Volume / quick win: `impressions_90d`

This measures how much search visibility a page already has. The table checks whether decline rates differ by the amount of traffic at stake. Observed pattern: the middle-volume buckets have higher observed rates (61.5% for 300–2,999 and 58.6% for 3,000–29,999) than the lowest- and highest-volume buckets (45.4% and 46.2%).

Verdict: ______

### Signal test 3 — Search ranking: `avg_position`

This measures average Google Search position; lower positive values are better. The table checks whether decline rates differ across ranking ranges. `avg_position == 0` is reported separately as **no position data**, not as a position-zero rank. Observed pattern: among rows with position data, the 11–20 range has the highest observed rate (61.0%); the no-position-data group is a measurement state, so it should not be interpreted as a real ranking tier.

Verdict: ______

In [ ]:
# The starter-data label is used only to calculate descriptive decline rates.
# It is not a signal feature.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

def print_bucket_table(frame, bucket_column, title, bucket_order):
    table = (
        frame.groupby(bucket_column, observed=False)["is_declining_label"]
        .agg(n="size", decline_count="sum", decline_rate="mean")
        .reindex(bucket_order)
        .reset_index()
        .rename(columns={bucket_column: "bucket"})
    )
    table["decline_rate"] = (table["decline_rate"] * 100).round(1)
    print(f"\n{title}")
    print(table.to_string(index=False))

# 1. Staleness / refresh: readable update-age ranges.
staleness_order = ["0-30 days", "31-90 days", "91-180 days", "181+ days"]
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, float("inf")],
    labels=staleness_order,
)
print_bucket_table(
    df, "staleness_bucket", "1. Staleness / refresh signal", staleness_order
)

# 2. Volume / quick win: the traffic bands are intentionally broad for a heavy-tailed metric.
volume_order = [
    "1-299 impressions", "300-2,999 impressions",
    "3,000-29,999 impressions", "30,000+ impressions",
]
df["volume_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 299, 2_999, 29_999, float("inf")],
    labels=volume_order,
    include_lowest=True,
)
print_bucket_table(df, "volume_bucket", "2. Volume / quick-win signal", volume_order)

# 3. Search ranking: zero is a distinct no-position-data state, never rank zero.
position_order = [
    "no position data (0)", "top 3 (1-3)", "page 1 (4-10)",
    "striking distance (11-20)", "pages 3-5 (21-50)", "deep (51+)",
]
position_with_data = pd.cut(
    df["avg_position"].where(df["avg_position"].ne(0)),
    bins=[0, 3, 10, 20, 50, float("inf")],
    labels=position_order[1:],
    include_lowest=True,
)
df["position_bucket"] = pd.Categorical(
    position_with_data.astype(object).where(
        df["avg_position"].ne(0), "no position data (0)"
    ),
    categories=position_order,
    ordered=True,
)
print_bucket_table(df, "position_bucket", "3. Search-ranking signal", position_order)

print("\nReview the observed tables, then write one permitted verdict in the markdown cell above.")


## 3. The flag-linked test

### Staleness / refresh flag assumption

**What is being tested?** FlyRank's staleness / refresh flag assumes that pages left unupdated for longer may have a higher observed decline rate. This comparison defines **Not stale** as fewer than 91 days since update and **Stale** as 91 or more days.

**What did the data show?** The stale group has a 60.85% observed decline rate versus 51.20% for the not-stale group: a difference of **+9.65 percentage points**.

**How much support is there?** This provides **weak descriptive support** for the assumption: the stale group is higher in this starter-data comparison, but the table alone does not show that staleness caused the decline.

**Human-review limitation:** A page may be deliberately unchanged because it is evergreen or already accurate. The starter decline label is also a current-window proxy, not a future observed outcome, so a reviewer should inspect page freshness and business value before choosing a refresh action.

In [ ]:
# Use the documented starter-data label only for this descriptive comparison.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

staleness_flag = df["days_since_last_update"].ge(91)
flag_table = (
    df.assign(
        stale_status=staleness_flag.map(
            {False: "Not stale (<91 days)", True: "Stale (91+ days)"}
        )
    )
    .groupby("stale_status")["is_declining_label"]
    .agg(n="size", decline_count="sum", decline_rate="mean")
    .reindex(["Not stale (<91 days)", "Stale (91+ days)"])
    .reset_index()
)
flag_table["decline_rate"] = (flag_table["decline_rate"] * 100).round(2)

not_stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Not stale (<91 days)"), "decline_rate"
].iloc[0]
stale_rate = flag_table.loc[
    flag_table["stale_status"].eq("Stale (91+ days)"), "decline_rate"
].iloc[0]
rate_difference_pp = stale_rate - not_stale_rate

print("Staleness / refresh flag comparison")
print(flag_table.to_string(index=False))
print(f"\nStale minus not-stale decline rate: {rate_difference_pp:+.2f} percentage points")


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.